# 07 · Request types by age and Electronics cases

This notebook replicates the Request types by age and Electronics cases paragraphs of the report's Results section.

> Request types by age: Young machines run 8.0 tickets per 100 machine-months vs. 4.8 for older ones (1.66 times), with higher rates in nearly every request group. Within model, Electronics' share is 1.31 times as large on young machines (mostly Touchscreen) and Mechanical's 0.70 times. 80% of the 1,447 extra tickets come from young machines running above older ones of the same model. Touchscreen tickets cluster on 2022–23 installs (1.54 per 100, 3.9 times the 2024–26 rate).
>
> Electronics cases: Per machine-month, Electronics runs 1.71 times as often on young machines, near the 1.66 for all tickets. Touchscreen stands out at 4.2 times: of its 411 tickets on young machines, 344 are connector reseats or replacements, 402 are on the four 2.0 models (LVD-Remote-2.0, MV-2.0, LVRR-WP-2.0, and SV-48-2.0), and 315 are on 2022–23 installs. Its rate steps from 0.36 to 1.61 per 100 between Jan–Apr and May–Aug 2022 installs, which are nearly the same age, so the step points to a production, part, or installation change. The case list has all 1,605 unplanned Electronics cases on the fleet machines behind slide 4.

We also check two numbers that the report's Definitions give for terms these paragraphs use: the 15 comparison models behind "within model" and the 1,156 extra tickets behind the 80%. Three numbers need more of IPA's case export and installed-base list than Part A's tables keep (the 344 connector tickets and the step from 0.36 to 1.61), so we read them from the aggregate tables that the Electronics case list's builder writes.

**How it works.** `followups.py` (Part A of the follow-ups) counts the unplanned tickets on IPA's fleet machines by request group, Category, model, age group, and install year, compares young machines (under 4 years) with older ones, and splits young machines' extra tickets into a within-model part and a model-mix part. We recompute the counts, rates, ratios, and splits from its tables by age band, model, and install year, from the per-machine table of `fleet_exposure.py`, and from the per-ticket table `followups/rp_pairs.csv`. From the script's tables we read only the bootstrap intervals and permutation-test results, which need its resampling, and the within-model share ratios and excesses, which need each of the 15 comparison models' tickets by request group or Category (the committed outputs hold these for six models).

**Re-run:** `verification/fleet_exposure.py` (about 30 s), then `verification/followups.py` (about 20 s), then `viz/deliverables/build_electronics_cases.py --tables` (about 10 s). With `RERUN = True`, they rewrite their 21 `fleet/fx_*` files, their 23 `followups/` files (the `ag_*` and `rp_*` tables and `followups_summary.json`), and the builder's three `electronics/ec_*.csv` tables in a scratch copy of `verification/outputs/`, and the notebook compares each file with the committed one. With `--tables`, the builder runs its data checks against the `followups/` tables and writes only labels and counts, with no workbook. `docs/09_followups_protocol.md` (§4 and §12) and `docs/10_followups_results.md` (items 1 to 4, §2, and §6) describe this part of the analysis.

**Used as committed, without re-running:** `fu/fu_ticket_last10.csv` and `fu/fu_devices.csv`, the ticket-to-device map that `db_followup.py` wrote from the telemetry database, which the notebooks do not re-read. `followups.py` reads them to give each machine its telemetry type, which these paragraphs do not use. `fleet_exposure.py` reads `htm_device_exposure.csv`, which `high_ticket_machines.py` writes from the telemetry-database extracts (notebook 02 re-runs it), for docs/05's chronic outliers and its interim telemetry-type rates, which these paragraphs do not use either. Before any result, `followups.py` compares its run with committed outputs of `fleet_exposure.py` (re-run here first), `reproduce_sql.py`, `reproduce_cleanroom.py`, and `audit_case_logic.py`, and stops if any differs. The Electronics case list itself, which the builder writes with IPA's case numbers and notes, is never committed, and this notebook does not read it.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'fleet_exposure.py')
    R.rerun(OUT, 'followups.py')
    R.rerun(OUT, 'viz/deliverables/build_electronics_cases.py', '--tables')
R.compare(OUT)

fleet_exposure.py: exit status 0, 45 s
    27/27 checks pass


followups.py: exit status 0, 39 s
    67/67 checks pass; 39 s


viz/deliverables/build_electronics_cases.py --tables: exit status 0, 9 s
    checks pass: Electronics tables written to electronics/
47 files written: 47 identical.


,file written by the re-run,compared with the committed file
0,electronics/ec_counts.csv,identical
1,electronics/ec_edge.csv,identical
2,electronics/ec_subsets.csv,identical
3,fleet/fx_age_rates.csv,identical
4,fleet/fx_country_rates.csv,identical
5,fleet/fx_coverage_rates.csv,identical
6,fleet/fx_customer_lorenz.csv,identical
7,fleet/fx_customer_rates.csv,identical
8,fleet/fx_family_rates.csv,identical
9,fleet/fx_machine_flags.csv,identical


## Young machines: 8.0 tickets per 100 machine-months vs. 4.8 for older ones, 1.66 times

Young machines are under 4 years old at the midpoint of their installed time in the period (the age bands <1, 1–2, and 2–4), and the rate is unplanned tickets per 100 installed machine-months. We add up each machine's tickets from `fleet/fx_machine_flags.csv`, which has one row per fleet machine with a ticket, and the machine-months from `followups/ag_support.csv`, which has one row per model and age group and also counts the machines without a ticket.

In [3]:
import numpy as np

summary = R.read_json(OUT, 'followups/followups_summary.json')
definitions = summary['definitions']
YOUNG_BANDS = definitions['young_bands']  # '<1', '1-2' and '2-4': under 4 years
AGES = ['Under 4', '4+']


def age_group(bands):
    """The age group of each age band."""
    return np.where(pd.Series(bands).isin(YOUNG_BANDS), AGES[0], AGES[1])


machines = R.read_csv(OUT, 'fleet/fx_machine_flags.csv')  # one row per fleet machine with an unplanned ticket
machines['age_group'] = age_group(machines.age_band)
support = R.read_csv(OUT, 'followups/ag_support.csv')  # one row per model and age group, over every fleet machine

by_age = pd.DataFrame({
    'machines': support.groupby('age_group').machines.sum(),
    'machines with a ticket': machines.groupby('age_group').size(),
    'installed machine-months': support.groupby('age_group').months.sum(),
    'unplanned tickets': machines.groupby('age_group').tickets.sum()}).reindex(AGES)
by_age['per 100 machine-months'] = 100 * by_age['unplanned tickets'] / by_age['installed machine-months']
rate_young, rate_old = by_age['per 100 machine-months']
months_young, months_old = by_age['installed machine-months']
ratio_all = rate_young / rate_old

listed = support.set_index(['model', 'age_group']).tickets
per_model = machines.groupby(['model', 'age_group']).tickets.sum()
print(f"{int(by_age['unplanned tickets'].sum()):,} unplanned tickets on {int(by_age.machines.sum()):,} fleet machines. "
      "The per-machine tickets add up to ag_support.csv's for every model and age group: "
      f'{bool(per_model.reindex(listed.index, fill_value=0).eq(listed).all() and per_model.sum() == listed.sum())}')
print(f'Young vs. older rate: {ratio_all:.4f}')
by_age.round(2)

6,750 unplanned tickets on 8,946 fleet machines. The per-machine tickets add up to ag_support.csv's for every model and age group: True
Young vs. older rate: 1.6633


,machines,machines with a ticket,installed machine-months,unplanned tickets,per 100 machine-months
age_group,,,,,
Under 4,3987,1778,45103.38,3629,8.05
4+,4959,1679,64517.91,3121,4.84


## Higher rates in nearly every request group

`followups/ag_band_rates.csv` has each request group's tickets and the machine-months in each of the seven age bands, so we add up the three young bands and the four older ones and divide, all models together. The 95% intervals of the ratios come from the script's bootstrap over accounts (2,000 draws), so we read them from `followups/ag_type_rates.csv`. We read "nearly every" as every one of the ten groups having a higher rate on young machines, with the interval excluding 1 for all groups but two.

In [4]:
bands = R.read_csv(OUT, 'followups/ag_band_rates.csv')  # one row per age band and request group
bands['age_group'] = age_group(bands.age_band)
GROUPS = definitions['groups']  # the ten request groups
ALL = 'All unplanned tickets'

sums = bands.groupby(['type', 'age_group'])[['tickets', 'months']].sum()
young_sums, old_sums = sums.xs(AGES[0], level='age_group'), sums.xs(AGES[1], level='age_group')
crude = pd.DataFrame({'young tickets': young_sums.tickets, 'older tickets': old_sums.tickets,
                      'young per 100': 100 * young_sums.tickets / young_sums.months,
                      'older per 100': 100 * old_sums.tickets / old_sums.months}).reindex(GROUPS + [ALL])
crude['ratio'] = crude['young per 100'] / crude['older per 100']

rates = R.read_csv(OUT, 'followups/ag_type_rates.csv')  # crude rates by request type, with the bootstrap intervals
boot = rates[rates.level.isin(['group', 'all'])].set_index('type').loc[crude.index]
crude['95% interval'] = [R.span(lo, hi, 2) for lo, hi in boot[['rate_ratio_lo', 'rate_ratio_hi']].to_numpy()]

lower = boot.rate_ratio_lo.loc[GROUPS]
n_groups, n_higher, n_clear = len(GROUPS), int((crude.ratio.loc[GROUPS] > 1).sum()), int((lower > 1).sum())
not_clear = list(lower.index[lower <= 1])


def how_many(higher, clear, total):
    """The report's words for how many groups run higher on young machines: 'every' when every interval excludes 1,
    'nearly every' when every estimate is higher and the interval excludes 1 for all but at most two."""
    if clear == total:
        return 'every'
    if higher == total and total - clear <= 2:
        return 'nearly every'
    return f'{clear} of {total}'


groups_word = how_many(n_higher, n_clear, n_groups)
print(f'{n_higher} of {n_groups} request groups have a higher rate on young machines, and the 95% interval excludes '
      f"1 for {n_clear}: all but {' and '.join(not_clear)}.")
print(f"All tickets from the age bands give the same ratio as above: {bool(np.isclose(crude.at[ALL, 'ratio'], ratio_all))}")
crude.round(3)

10 of 10 request groups have a higher rate on young machines, and the 95% interval excludes 1 for 8: all but Network and Software.
All tickets from the age bands give the same ratio as above: True


,young tickets,older tickets,young per 100,older per 100,ratio,95% interval
type,,,,,,
Door,1193,987,2.645,1.530,1.729,1.49–2.01
Mechanical,714,623,1.583,0.966,1.639,1.33–2.04
Electronics,875,730,1.940,1.131,1.715,1.43–2.10
Network,193,250,0.428,0.387,1.104,0.83–1.45
Software,41,47,0.091,0.073,1.248,0.68–2.28
Operator error,84,61,0.186,0.095,1.970,1.33–2.93
Other,84,60,0.186,0.093,2.003,1.31–3.05
No Trouble Found,265,230,0.588,0.356,1.648,1.32–2.05
Monitoring,80,37,0.177,0.057,3.093,1.94–5.19


## Within model: Electronics' share 1.31 times as large on young machines and Mechanical's 0.70 times

A group's share ratio within model is its share of a model's young-machine tickets ÷ its share of the model's older-machine tickets, pooled by Mantel–Haenszel over the 15 comparison models, those with 100+ installed machine-months in each age group, which we count in `ag_support.csv`. The pooled ratio needs every comparison model's tickets by group, which the committed outputs hold for only six models (`followups/ag_by_model.csv`), so we read it, with its bootstrap interval and test result, from `followups/ag_type_adjusted.csv`. As a check, we apply the same formula to those six models and to the other nine models' tickets in `followups/rp_pairs.csv`, which lists every ticket opened before Aug 2nd, 2026, and the next ticket on its machine, and we place each ticket it misses wherever it moves the ratio most, which gives a range that must hold the script's ratio.

In [5]:
adjusted = R.read_csv(OUT, 'followups/ag_type_adjusted.csv')
primary = adjusted[adjusted.variant == 'primary']  # the comparison models: nine groups and their Categories
adj_groups = primary[primary.level == 'group'].set_index('type')
adj_cats = primary[primary.level == 'category'].set_index('type')

# The comparison models: at least 100 installed machine-months in each age group
months_by = support.pivot(index='model', columns='age_group', values='months').reindex(columns=AGES).fillna(0)
comparison = sorted(months_by.index[(months_by >= definitions['min_support_months']).all(axis=1)])
assert comparison == sorted(support.model[support.in_support].unique()) == sorted(summary['facts']['support']['support_models'])
family = support[support.model.isin(comparison)].pivot(index='model', columns='age_group', values='family_tickets')
n1, n0 = family[AGES[0]], family[AGES[1]]  # each model's young and older tickets in the nine groups
weight_young, weight_old = n0 / (n1 + n0), n1 / (n1 + n0)  # Mantel–Haenszel weights of a model's young and older counts
sr_el = adj_groups.at['Electronics', 'share_ratio_mh']
sr_mech = adj_groups.at['Mechanical', 'share_ratio_mh']

# Check: the same formula on the per-model counts the committed outputs show
six = R.read_csv(OUT, 'followups/ag_by_model.csv')  # every group's tickets on the six models with 30+ in each age group
pairs = R.read_csv(OUT, 'followups/rp_pairs.csv')  # one row per ticket opened before Aug 2nd, 2026, with the next one
COLS = ['case_id', 'machine_id', 'group', 'category']
seen = pd.concat([pairs[['case_id', 'machine_id', 'index_group', 'index_category']].set_axis(COLS, axis=1),
                  pairs.loc[pairs.next_case_id.notna(), ['next_case_id', 'machine_id', 'next_group', 'next_category']]
                  .set_axis(COLS, axis=1)]).drop_duplicates('case_id')
seen = seen.merge(machines[['serial_id', 'model', 'age_group', 'install_year']], left_on='machine_id',
                  right_on='serial_id')  # fleet machines only; model, age group and install year from IPA's list
assert set(six.model) <= set(comparison)
assert (seen.groupby('machine_id').size().reindex(machines.serial_id, fill_value=0).to_numpy()
        <= machines.tickets.to_numpy()).all()  # never more tickets than the machine has
others = [m for m in comparison if m not in set(six.model)]
cells_other = pd.MultiIndex.from_product([others, AGES], names=['model', 'age_group'])
seen_other = seen[seen.model.isin(others)]
missing = listed.reindex(cells_other, fill_value=0) - seen_other.groupby(['model', 'age_group']).size().reindex(
    cells_other, fill_value=0)  # the nine models' tickets that rp_pairs.csv does not show
assert (missing >= 0).all()


def share_ratio_range(group):
    """The lowest and highest share ratio the counts we can see allow, with every ticket we cannot see placed in the
    model that moves the ratio most."""
    young, older = pd.Series(0.0, index=comparison), pd.Series(0.0, index=comparison)
    by_six = six.pivot(index='model', columns='age_group', values=f'tickets_{group}')
    young.loc[by_six.index], older.loc[by_six.index] = by_six[AGES[0]].to_numpy(), by_six[AGES[1]].to_numpy()
    counts = seen_other[seen_other.group == group].groupby(['model', 'age_group']).size()
    for m in others:
        young[m], older[m] = counts.get((m, AGES[0]), 0), counts.get((m, AGES[1]), 0)
    left_young = adj_groups.at[group, 'under4_tickets'] - young.sum()
    left_old = adj_groups.at[group, 'plus4_tickets'] - older.sum()
    assert 0 <= left_young <= missing.xs(AGES[0], level='age_group').sum()
    assert 0 <= left_old <= missing.xs(AGES[1], level='age_group').sum()
    wy = weight_young[[m for m in others if missing[(m, AGES[0])] > 0]]
    wo = weight_old[[m for m in others if missing[(m, AGES[1])] > 0]]
    num, den = (young * weight_young).sum(), (older * weight_old).sum()
    lo = (num + (left_young * wy.min() if left_young else 0)) / (den + (left_old * wo.max() if left_old else 0))
    hi = (num + (left_young * wy.max() if left_young else 0)) / (den + (left_old * wo.min() if left_old else 0))
    return int(left_young), int(left_old), lo, hi


passing = adj_groups[adj_groups.significant]
print(f"Groups that pass the within-model test: {' and '.join(passing.index)} "
      f"(Benjamini–Hochberg q = {', '.join(f'{q:.4f}' for q in passing.q_bh)}).")
rows = {}
for group in ('Electronics', 'Mechanical'):
    row = adj_groups.loc[group]
    left_young, left_old, lo, hi = share_ratio_range(group)
    print(f'{group}: the counts we can see, with {left_young} young and {left_old} older tickets placed, allow '
          f"{lo:.4f} to {hi:.4f}, and the script's {row.share_ratio_mh:.4f} lies inside: "
          f'{bool(lo <= row.share_ratio_mh <= hi)}')
    rows[group] = {'young tickets': int(row.under4_tickets), 'older tickets': int(row.plus4_tickets),
                   'young share': R.pct(row.under4_tickets / n1.sum(), 1),
                   'older share': R.pct(row.plus4_tickets / n0.sum(), 1),
                   'share ratio within model': R.num(row.share_ratio_mh, 3),
                   '95% interval': R.span(row.share_ratio_lo, row.share_ratio_hi, 2),
                   'range from the counts we can see': R.span(lo, hi, 2)}
pd.DataFrame(rows).T

Groups that pass the within-model test: Mechanical and Electronics (Benjamini–Hochberg q = 0.0022, 0.0022).
Electronics: the counts we can see, with 2 young and 5 older tickets placed, allow 1.2854 to 1.3225, and the script's 1.3132 lies inside: True


Mechanical: the counts we can see, with 2 young and 6 older tickets placed, allow 0.6839 to 0.7023, and the script's 0.6988 lies inside: True


,young tickets,older tickets,young share,older share,share ratio within model,95% interval,range from the counts we can see
Electronics,870,465,25.0%,22.4%,1.313,1.09–1.62,1.29–1.32
Mechanical,690,406,19.8%,19.6%,0.699,0.59–0.84,0.68–0.70


## Electronics' larger share is mostly Touchscreen

"Mostly Touchscreen" refers to Electronics' excess within model: young machines' Electronics tickets minus those expected at the same within-model mix, the numerator of the script's Cochran–Mantel–Haenszel statistic. Like the share ratio, it needs every comparison model's tickets by Category, so we read each Electronics Category's excess from `ag_type_adjusted.csv` (the Categories with 20+ tickets from 10+ accounts in each age group) and find the one that gives most of the group's. The section on the 402 tickets below recomputes Touchscreen's own excess from per-model counts.

In [6]:
excess_el = adj_groups.at['Electronics', 'excess_under4_tickets']
el_excess = adj_cats[(adj_cats.group == 'Electronics') & adj_cats.estimable].excess_under4_tickets
top_share = el_excess.max() / excess_el
mostly = el_excess.idxmax() if top_share > 0.5 else 'no single Category'
print(f"{el_excess.idxmax()} gives {el_excess.max():+.1f} of Electronics' {excess_el:+.1f} excess young-machine tickets "
      f'({R.pct(top_share)}).')
pd.DataFrame({'excess young tickets': el_excess.round(1),
              'share ratio within model': adj_cats.share_ratio_mh[el_excess.index].round(3)}).sort_values(
    'excess young tickets', ascending=False)

Touchscreen gives +44.3 of Electronics' +49.3 excess young-machine tickets (90%).


,excess young tickets,share ratio within model
type,,
Touchscreen,44.3,1.853
Power Supply,6.7,1.581
Badge Reader,-3.9,0.772
Computer Board,-6.0,0.911


## 80% of the 1,447 extra tickets come from young machines running above older ones of the same model

Extra tickets are young machines' tickets minus their machine-months times the older machines' rate, and they split exactly into a within-model part (on the comparison models, young machines' tickets minus their machine-months times the same model's older-machine rate), a model-mix part, and a part for the other models. `ag_support.csv` has the tickets and machine-months by model and age group that the split needs, and we compare the parts with `followups/ag_decomposition.csv` and `followups_summary.json`.

In [7]:
tickets_by = support.pivot(index='model', columns='age_group', values='tickets').reindex(columns=AGES).fillna(0)
inside = tickets_by.index.isin(comparison)
old_rate = tickets_by[AGES[1]].sum() / months_by[AGES[1]].sum()  # older machines' rate, all models
own_rate = tickets_by.loc[inside, AGES[1]] / months_by.loc[inside, AGES[1]]  # each comparison model's older-machine rate
expected_same_model = (months_by.loc[inside, AGES[0]] * own_rate).sum()

extra = tickets_by[AGES[0]].sum() - months_by[AGES[0]].sum() * old_rate
within = tickets_by.loc[inside, AGES[0]].sum() - expected_same_model
mix = expected_same_model - months_by.loc[inside, AGES[0]].sum() * old_rate
outside = tickets_by.loc[~inside, AGES[0]].sum() - months_by.loc[~inside, AGES[0]].sum() * old_rate
share_within = within / extra

dec = R.read_csv(OUT, 'followups/ag_decomposition.csv')
dec_all = dec[dec.level == 'all'].iloc[0]
part_a = summary['facts']['part_a']
ours = [extra, within, mix, outside]
print('The parts add up to the extra tickets: '
      f'{bool(np.isclose(within + mix + outside, extra))}. They match ag_decomposition.csv: '
      f"{bool(np.allclose(ours, dec_all[['extra_tickets', 'within_model_age_tickets', 'model_mix_tickets', 'outside_support_tickets']].astype(float)))}"
      ', and followups_summary.json: '
      f"{bool(np.allclose(ours, [part_a[k] for k in ('extra_tickets', 'extra_within_model_age', 'extra_model_mix', 'extra_outside_support')]))}")
pd.DataFrame({'tickets': [R.num(x, 1) for x in ours], 'share of the extra': [R.pct(x / extra, 1) for x in ours]},
             index=['extra tickets on young machines', 'within model: young above older machines of the same model',
                    'model mix: young machines more often on high-rate models',
                    f'models outside the {len(comparison)} comparison models'])

The parts add up to the extra tickets: True. They match ag_decomposition.csv: True, and followups_summary.json: True


,tickets,share of the extra
extra tickets on young machines,"1,447.2",100.0%
within model: young above older machines of the same model,"1,156.1",79.9%
model mix: young machines more often on high-rate models,282.2,19.5%
models outside the 15 comparison models,8.9,0.6%


## Touchscreen tickets cluster on 2022–23 installs: 1.54 per 100, 3.9 times the 2024–26 rate

`followups/ag_install_year.csv` counts Touchscreen tickets and machine-months by comparison model and install year, at any age, so we add up the years into the report's four install cohorts (the cohort labels of `followups/ag_install_pooled.csv`) and divide. The cohorts were chosen after seeing the data, so the result is descriptive, and we compare our sums with the pooled table.

In [8]:
years = R.read_csv(OUT, 'followups/ag_install_year.csv')  # three Categories by comparison model and install year
ts_years = years[years.category == 'Touchscreen']
pooled = R.read_csv(OUT, 'followups/ag_install_pooled.csv')
pooled_ts = pooled[(pooled.category == 'Touchscreen') & (pooled.unit == 'cohort')].set_index('install')
COHORTS = list(pooled_ts.index)


def cohort_years(label):
    """'2022-23' -> (2022, 2023)."""
    first, last = label.split('-')
    return int(first), int(first[:2] + last)


def dashed(label):
    """'2022-23' -> '2022–23', as the report prints a span of years."""
    return label.replace('-', '–')


cohorts = pd.DataFrame([ts_years[ts_years.install_year.between(*cohort_years(c))][
    ['machines', 'under4_machines', 'months', 'tickets']].sum().rename(c) for c in COHORTS])
cohorts = cohorts.astype({'machines': int, 'under4_machines': int, 'tickets': int})
cohorts['per 100 machine-months'] = 100 * cohorts.tickets / cohorts.months
newest = COHORTS[-1]
cohorts[f'vs. {dashed(newest)}'] = cohorts['per 100 machine-months'] / cohorts.at[newest, 'per 100 machine-months']
cohorts['share of machines under 4'] = cohorts.under4_machines / cohorts.machines
cluster = cohorts['per 100 machine-months'].idxmax()
rate_cluster = cohorts.at[cluster, 'per 100 machine-months']
ratio_cluster = cohorts.at[cluster, f'vs. {dashed(newest)}']

same = (np.allclose(cohorts[['machines', 'months', 'tickets', 'per 100 machine-months']].to_numpy(float),
                    pooled_ts.loc[COHORTS, ['machines', 'months', 'tickets', 'rate_per_100']].to_numpy(float))
        and np.allclose(cohorts[f'vs. {dashed(newest)}'], pooled_ts.loc[COHORTS, 'ratio_to_newest_cohort']))
print(f'Highest Touchscreen rate: {dashed(cluster)} installs, {rate_cluster:.3f} per 100 machine-months, '
      f'{ratio_cluster:.3f} times the {dashed(newest)} rate. Our cohort sums match ag_install_pooled.csv: {same}')
cohorts.drop(columns='under4_machines').rename(index=dashed).round(3)

Highest Touchscreen rate: 2022–23 installs, 1.538 per 100 machine-months, 3.895 times the 2024–26 rate. Our cohort sums match ag_install_pooled.csv: True


,machines,months,tickets,per 100 machine-months,vs. 2024–26,share of machines under 4
2016–18,1003,13049.298,27,0.207,0.524,0.000
2019–21,1971,25643.236,68,0.265,0.671,0.000
2022–23,1599,20803.417,320,1.538,3.895,0.954
2024–26,2377,24309.487,96,0.395,1.000,1.000


## Electronics runs 1.71 times as often on young machines, near the 1.66 for all tickets

These are the crude rate ratios per installed machine-month of the Electronics group and of all unplanned tickets, all models together, from the band sums of `ag_band_rates.csv` above. Electronics' share of tickets is nearly the same at both ages, which is why its ratio is close to that of all tickets.

In [9]:
ratio_el = crude.at['Electronics', 'ratio']
share_el_young = crude.at['Electronics', 'young tickets'] / crude.at[ALL, 'young tickets']
share_el_old = crude.at['Electronics', 'older tickets'] / crude.at[ALL, 'older tickets']
print(f'Electronics: {ratio_el:.4f} times per machine-month vs. {ratio_all:.4f} for all tickets. Electronics is '
      f"{R.pct(share_el_young, 1)} of young machines' tickets vs. {R.pct(share_el_old, 1)} of older machines'.")

Electronics: 1.7146 times per machine-month vs. 1.6633 for all tickets. Electronics is 24.1% of young machines' tickets vs. 23.4% of older machines'.


## Touchscreen stands out at 4.2 times, with 411 tickets on young machines

`followups/ag_type_rates.csv` counts each Electronics Category's tickets on young and older machines, all models together, and we divide them by the machine-months above. Touchscreen stands out when its ratio is the highest of the Electronics Categories with at least 30 tickets in each age group (the script's threshold for showing a model or type).

In [10]:
el_rows = rates[(rates.level == 'category') & (rates.group == 'Electronics')].set_index('type')
el_cat = pd.DataFrame({'young tickets': el_rows.under4_tickets, 'older tickets': el_rows.plus4_tickets})
el_cat['young per 100'] = 100 * el_cat['young tickets'] / months_young
el_cat['older per 100'] = 100 * el_cat['older tickets'] / months_old
el_cat['ratio'] = el_cat['young per 100'] / el_cat['older per 100']

min_tickets = definitions['min_shown_tickets']
shown_cats = el_cat[(el_cat['young tickets'] >= min_tickets) & (el_cat['older tickets'] >= min_tickets)]
stands_out = shown_cats.ratio.idxmax()
rest = shown_cats.drop(stands_out)
ratio_ts = el_cat.at['Touchscreen', 'ratio']
ts_young = int(el_cat.at['Touchscreen', 'young tickets'])
print(f'{stands_out}: {shown_cats.ratio.max():.3f} times. The other Electronics Categories with {min_tickets}+ tickets '
      f'in each age group: {R.span(rest.ratio.min(), rest.ratio.max(), 2)} times.')
shown_cats.round(3)

Touchscreen: 4.230 times. The other Electronics Categories with 30+ tickets in each age group: 0.97–1.75 times.


,young tickets,older tickets,young per 100,older per 100,ratio
type,,,,,
Computer Board,259,383,0.574,0.594,0.967
Touchscreen,411,139,0.911,0.215,4.230
Power Supply,83,106,0.184,0.164,1.120
Badge Reader,60,49,0.133,0.076,1.752


## 344 connector reseats or replacements

The 344 are the young-machine Touchscreen tickets with Sub Category Connector Reseat/Replace. Part A's tables have no Sub Category, so the case-list builder counts them from IPA's case export and writes them, with the other subsets of its Summary (table 7), to `electronics/ec_subsets.csv`. Its count of young-machine Touchscreen tickets equals the 411 above.

In [11]:
subsets = R.read_csv(OUT, 'electronics/ec_subsets.csv').set_index('key')
assert subsets.at['u4_focus', 'tickets'] == ts_young
n_conn = int(subsets.at['u4_conn', 'tickets'])
print(f"{n_conn} of the {ts_young} young-machine Touchscreen tickets are "
      f"{subsets.at['u4_conn', 'label'].removeprefix('of which ')}.")
subsets.loc[['u4', 'u4_focus', 'u4_conn', 'u4_coh'], ['label', 'tickets', 'machines', 'accounts']]

344 of the 411 young-machine Touchscreen tickets are Connector Reseat/Replace.


,label,tickets,machines,accounts
key,,,,
u4,"Electronics, under 4",875,640,316
u4_focus,"Touchscreen, under 4",411,313,184
u4_conn,of which Connector Reseat/Replace,344,268,163
u4_coh,of which installed 2022-23,315,232,125


## 402 on the four 2.0 models and 315 on 2022–23 installs

`ag_install_year.csv` counts Touchscreen tickets by comparison model and install year, and every install year but 2022 has only young or only older machines, so only the 2022 counts need splitting by age: we count the older machines' Touchscreen tickets on 2022 installs in `rp_pairs.csv` (each machine's age and install year from `fx_machine_flags.csv`) and check that this count is complete. We also check that all 411 young-machine Touchscreen tickets are on the comparison models and that the per-model counts give the script's within-model share ratio and excess for Touchscreen. The four 2.0 models are the scrubEx family's models whose names end in 2.0 (`fleet/fx_model_rates.csv`).

In [12]:
# The one install year whose machines are partly young and partly older
mixed = ts_years.install_year[(ts_years.under4_machines > 0) & (ts_years.under4_machines < ts_years.machines)].unique()
assert len(mixed) == 1
MIXED = int(mixed[0])
before, after = ts_years.install_year < MIXED, ts_years.install_year > MIXED
assert (ts_years.under4_machines[before] == 0).all() and (ts_years.under4_machines[after] == ts_years.machines[after]).all()

# Older machines' Touchscreen tickets on that year's installs, ticket by ticket, and a check that none is missing
ts_seen = seen[(seen.category == 'Touchscreen') & seen.model.isin(comparison)]
old_mixed = ts_seen[(ts_seen.install_year == MIXED) & (ts_seen.age_group == AGES[1])].groupby('model').size()
complete = int(old_mixed.sum()) == adj_cats.at['Touchscreen', 'plus4_tickets'] - ts_years.tickets[before].sum()

assert sorted(ts_years.model.unique()) == comparison
old_mixed = old_mixed.reindex(comparison, fill_value=0)


def tickets_per_model(rows):
    return ts_years[rows].groupby('model').tickets.sum().reindex(comparison, fill_value=0)


first, last = cohort_years(cluster)
assert first <= MIXED <= last and last + 1 == cohort_years(newest)[0]
assert cohort_years(newest)[1] >= ts_years.install_year.max()
by_model = pd.DataFrame({
    'on older machines': tickets_per_model(before) + old_mixed,
    f'young, {dashed(cluster)} installs': tickets_per_model(ts_years.install_year.between(first, last)) - old_mixed,
    f'young, {dashed(newest)} installs': tickets_per_model(ts_years.install_year.between(*cohort_years(newest))),
    'on young machines': tickets_per_model(ts_years.install_year >= MIXED) - old_mixed})
assert (by_model['on young machines'] == by_model.iloc[:, 1] + by_model.iloc[:, 2]).all()

# The 2.0 models: the scrubEx family's models whose names end in 2.0
model_rates = R.read_csv(OUT, 'fleet/fx_model_rates.csv')  # one row per model, with its family
v2 = sorted(model_rates.model[(model_rates.family == 'scrubEx') & model_rates.model.str.endswith('2.0')])
v2_order = by_model.loc[v2, 'on young machines'].sort_values(ascending=False).index.tolist()
n_v2 = int(by_model.loc[v2, 'on young machines'].sum())
n_cluster = int(by_model[f'young, {dashed(cluster)} installs'].sum())
n_young_ts = int(by_model['on young machines'].sum())

# The per-model counts give the script's within-model share ratio and excess for Touchscreen
a, b = by_model['on young machines'], by_model['on older machines']
sr_ts = (a * weight_young).sum() / (b * weight_old).sum()
excess_ts = (a - n1 * (a + b) / (n1 + n0)).sum()
print(f"{MIXED} is the only install year that mixes ages. The count of older machines' Touchscreen tickets on {MIXED} "
      f'installs ({int(old_mixed.sum())}) is complete: {complete}')
print(f'All {ts_young} young-machine Touchscreen tickets (ag_type_rates.csv) are on the comparison models: '
      f'{n_young_ts == ts_young}')
builder = R.read_csv(OUT, 'electronics/ec_counts.csv').set_index('key').value
assert builder['focus_under4_core_models'] == n_v2 and subsets.at['u4_coh', 'tickets'] == n_cluster
print(f"The case-list builder's counts from IPA's case export agree: {n_v2} on the 2.0 models and {n_cluster} on "
      f'{dashed(cluster)} installs.')
print(f'Within-model share ratio {sr_ts:.4f} and excess {excess_ts:+.2f} match ag_type_adjusted.csv: '
      f"{bool(np.isclose(sr_ts, adj_cats.at['Touchscreen', 'share_ratio_mh']) and np.isclose(excess_ts, adj_cats.at['Touchscreen', 'excess_under4_tickets']))}")

WORDS = {1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine'}


def and_join(names):
    """['a', 'b', 'c'] -> 'a, b, and c'."""
    return ' and '.join(names) if len(names) < 3 else ', '.join(names[:-1]) + f', and {names[-1]}'


table = by_model[by_model['on young machines'] > 0].sort_values('on young machines', ascending=False)
table.insert(0, 'scrubEx 2.0 model', np.where(table.index.isin(v2), 'yes', 'no'))
table.loc[f'All {len(comparison)} comparison models'] = ['', *by_model.sum()]
table

2022 is the only install year that mixes ages. The count of older machines' Touchscreen tickets on 2022 installs (5) is complete: True
All 411 young-machine Touchscreen tickets (ag_type_rates.csv) are on the comparison models: True
The case-list builder's counts from IPA's case export agree: 402 on the 2.0 models and 315 on 2022–23 installs.
Within-model share ratio 1.8530 and excess +44.32 match ag_type_adjusted.csv: True


,scrubEx 2.0 model,on older machines,"young, 2022–23 installs","young, 2024–26 installs",on young machines
model,,,,,
LVD-Remote-2.0,yes,32,141,37,178
MV-2.0,yes,11,109,29,138
LVRR-WP-2.0,yes,18,53,21,74
SV-48-2.0,yes,1,8,4,12
alEx LC-D,no,1,1,2,3
alEx MC-S,no,0,1,1,2
alEx MC-D,no,0,0,2,2
A-LC,no,2,1,0,1
alEx SD 2.0,no,1,1,0,1


## The step from 0.36 to 1.61 per 100 between Jan–Apr and May–Aug 2022 installs

The step compares the comparison models' machines installed January to April 2022 with those installed May to August 2022, by their Touchscreen tickets at any age per 100 installed machine-months. Part A's tables give install dates only as years, so the case-list builder counts each window from IPA's installed-base list and writes the counts to `electronics/ec_edge.csv`, and we recompute the rates from them. The builder checks that its counts by install month add up, for 2022, to `followups/ag_install_pooled.csv`, and here we check that the two windows fit inside that 2022 row. The windows' install dates are four months apart, so their machines are nearly the same age. The 4-year line falls inside the first window, which is why only some of its machines count as young.

In [13]:
edge = R.read_csv(OUT, 'electronics/ec_edge.csv')
edge['per 100'] = 100 * edge.tickets / edge.months
assert np.allclose(edge['per 100'], edge.rate_per_100)
rate_early, rate_late = edge['per 100']
year = R.read_csv(OUT, 'followups/ag_install_pooled.csv', dtype={'install': str}).query(
    "category == 'Touchscreen' and unit == 'year' and install == '2022'").iloc[0]
assert edge.machines.sum() <= year.machines and edge.months.sum() <= year.months and edge.tickets.sum() <= year.tickets


def short(label):
    """'January to April 2022' -> ('Jan–Apr', '2022')."""
    first, last = label.split(' to ')
    return f'{first[:3]}–{last[:3]}', last.split()[-1]


(early, y0), (late, y1) = map(short, edge.label)
windows = f'{early} and {late} {y1}' if y0 == y1 else f'{early} {y0} and {late} {y1}'
print(f'Touchscreen tickets per 100 machine-months: {rate_early:.2f} on {edge.label[0]} installs and '
      f'{rate_late:.2f} on {edge.label[1]} installs ({rate_late / rate_early:.1f} times).')
edge[['label', 'machines', 'under4', 'months', 'tickets', 'per 100']].round(2)

Touchscreen tickets per 100 machine-months: 0.36 on January to April 2022 installs and 1.61 on May to August 2022 installs (4.4 times).


,label,machines,under4,months,tickets,per 100
0,January to April 2022,232,158,3018.38,11,0.36
1,May to August 2022,253,253,3291.60,53,1.61


## All 1,605 unplanned Electronics cases on the fleet machines behind slide 4

The case list holds every unplanned ticket in the Electronics request group on the fleet machines of Part A (slide 4's population), one row per case. It is not committed, so we count those tickets over the seven age bands of `ag_band_rates.csv` and compare the count with `followups/ag_by_status.csv`, which splits the same tickets by service status.

In [14]:
el_bands = bands[bands.type == 'Electronics']
n_el = int(el_bands.tickets.sum())
n_el_young = int(el_bands.tickets[el_bands.age_group == AGES[0]].sum())
status = R.read_csv(OUT, 'followups/ag_by_status.csv')  # the same tickets by service status and age group
print(f'{n_el:,} unplanned Electronics tickets on the {int(el_bands.machines.sum()):,} fleet machines: {n_el_young:,} on '
      f'young machines and {n_el - n_el_young:,} on older ones. ag_by_status.csv gives the same total: '
      f"{n_el == int(status.tickets[status.type == 'Electronics'].sum())}")
assert builder['tickets'] == n_el and builder['under4'] == n_el_young
print(f"The case-list builder lists the same {n_el:,} cases ({builder['under4']:,} on young machines).")

1,605 unplanned Electronics tickets on the 8,946 fleet machines: 875 on young machines and 730 on older ones. ag_by_status.csv gives the same total: True
The case-list builder lists the same 1,605 cases (875 on young machines).


## Check against the report

In [15]:
R.check([
    ('Young machines: unplanned tickets per 100 machine-months', '8.0', R.num(rate_young, 1)),
    ('Older machines: unplanned tickets per 100 machine-months', '4.8', R.num(rate_old, 1)),
    ('Young vs. older rate, all tickets', '1.66', R.num(ratio_all, 2)),
    ('Request groups with a higher rate on young machines', 'nearly every', groups_word),
    ("Within model: Electronics' share on young vs. older machines", '1.31', R.num(sr_el, 2)),
    ("Category behind most of Electronics' larger share", 'Touchscreen', mostly),
    ("Within model: Mechanical's share on young vs. older machines", '0.70', R.num(sr_mech, 2)),
    ('Share of the extra tickets from young machines above older ones of the same model', '80%', R.pct(share_within)),
    ('Extra tickets on young machines', '1,447', R.num(extra)),
    ('Install cohort where Touchscreen tickets cluster', '2022–23', dashed(cluster)),
    ('Touchscreen tickets per 100 machine-months on those installs', '1.54', R.num(rate_cluster, 2)),
    ("That rate vs. the newest cohort's", '3.9', R.num(ratio_cluster, 1)),
    ('Newest install cohort', '2024–26', dashed(newest)),
    ('Electronics: young vs. older rate per machine-month', '1.71', R.num(ratio_el, 2)),
    ('All tickets: young vs. older rate (second paragraph)', '1.66', R.num(ratio_all, 2)),
    ('Touchscreen: young vs. older rate per machine-month', '4.2', R.num(ratio_ts, 1)),
    ('Electronics Category that stands out (highest ratio, 30+ tickets in each age group)', 'Touchscreen', stands_out),
    ('Touchscreen tickets on young machines', '411', R.num(ts_young)),
    ('Young-machine Touchscreen tickets that are connector reseats or replacements', '344', R.num(n_conn)),
    ('Young-machine Touchscreen tickets on the 2.0 models', '402', R.num(n_v2)),
    ('Number of 2.0 models (scrubEx models named 2.0)', 'four', WORDS[len(v2)]),
    ('The 2.0 models, most young-machine Touchscreen tickets first', 'LVD-Remote-2.0, MV-2.0, LVRR-WP-2.0, and SV-48-2.0',
     and_join(v2_order)),
    ('Young-machine Touchscreen tickets on 2022–23 installs', '315', R.num(n_cluster)),
    ('Touchscreen per 100 machine-months: earlier install window', '0.36', R.num(rate_early, 2)),
    ('Touchscreen per 100 machine-months: later install window', '1.61', R.num(rate_late, 2)),
    ('The two install windows', 'Jan–Apr and May–Aug 2022', windows),
    ('Unplanned Electronics cases on the fleet machines behind slide 4', '1,605', R.num(n_el)),
    ('Comparison models for within-model results (Definitions, Within model)', '15', R.num(len(comparison))),
    ('Extra tickets within model (Definitions, Extra tickets and the 80%)', '1,156', R.num(within)),
]);

,Finding,Report,Recomputed,Match
0,Young machines: unplanned tickets per 100 mach...,8.0,8.0,yes
1,Older machines: unplanned tickets per 100 mach...,4.8,4.8,yes
2,"Young vs. older rate, all tickets",1.66,1.66,yes
3,Request groups with a higher rate on young mac...,nearly every,nearly every,yes
4,Within model: Electronics' share on young vs. ...,1.31,1.31,yes
5,Category behind most of Electronics' larger share,Touchscreen,Touchscreen,yes
6,Within model: Mechanical's share on young vs. ...,0.70,0.70,yes
7,Share of the extra tickets from young machines...,80%,80%,yes
8,Extra tickets on young machines,"1,447","1,447",yes
9,Install cohort where Touchscreen tickets cluster,2022–23,2022–23,yes


All 29 values match the report.
